# Lab 7 · Kiểm chứng quy tắc xử lý chuỗi

**Lập trình xử lý dữ liệu · 2627-1 · Giờ thực hành · bài 7**

Chọn **Save a copy in Drive** trước khi sửa.

Lab dùng bảng nhỏ để kiểm quy tắc, rồi áp dụng lên `comments` và `amenities`
của Santiago, snapshot **29/06/2026**. Mục tiêu là tạo được kết quả có ý nghĩa,
không chỉ viết được một mẫu regex chạy thành công.

**Luyện tập bổ sung:** [20 bài luyện regex](https://colab.research.google.com/github/uet-iai-course/programming-for-data-processing/blob/main/2627-1/notebooks/lab-07-regex.ipynb), kèm code khởi đầu và bộ test để tự kiểm tra.

## Cách làm việc trong bài lab

- Điền các ô `TODO`; đọc kết quả và giải thích bằng dữ liệu gốc.
  `assert` kiểm một số điều kiện, không thay thế việc xem các trường hợp sai.
- Với các phần có hướng dẫn, tự viết trước khi nhờ AI.
  Các bài kiểm tra trên lớp (🚫 đóng) đánh giá các kỹ năng này.
- Bài tự làm ✅ mở cho phép dùng AI, kèm khai báo và cách kiểm chứng.
- Khi cần hỗ trợ, trao đổi với giảng viên thực hành về yêu cầu hoặc trường hợp chưa hiểu.

## Mục tiêu

1. Chuẩn hoá mà không làm mất thông tin; phân biệt giá trị thiếu từ nguồn với lỗi chuyển đổi.
2. Dùng quy tắc để tìm, kiểm tra định dạng và trích xuất đúng mục đích.
3. Phát hiện các trường hợp khớp nhầm trong văn bản thật.
4. Tính tỷ lệ tiện nghi theo chỗ ở, giữ đúng mẫu số.

## Chuẩn bị

Chạy hai ô dưới trước. Phiên bản pandas được ghim để khớp với ví dụ trên slide.
Nếu phiên Colab đã nạp pandas phiên bản khác, chọn **Restart session and run all** sau khi cài.

In [ ]:
%pip -q install pandas==3.0.3

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
print("pandas:", pd.__version__)

## Phần 0 · Khởi động

Hai bài nhỏ dưới đây dùng dữ liệu giả lập. Không tự gộp các cách viết ngoài quy tắc đã nêu.

In [ ]:
s = pd.Series(["  Wifi ", "wifi", "WIFI!", "wi-fi"])

# TODO: bỏ khoảng trắng ở hai đầu và đổi sang chữ thường.
s_chuan = ...

assert s_chuan.tolist() == ["wifi", "wifi", "wifi!", "wi-fi"]
assert s_chuan.index.equals(s.index)
print(s_chuan.value_counts())

Hai giá trị cuối vẫn khác `wifi`. Nếu muốn gộp chúng, cần nêu thêm quy tắc, không mặc định xoá mọi dấu câu.

In [ ]:
s = pd.Series(["a.b", "axb", None])

# TODO: tạo mask tìm đúng dấu chấm; dòng thiếu không được chọn.
mask = ...

assert mask.tolist() == [True, False, False]
print(s[mask].tolist())

## Phần 1 · Đổi giá sang số và giữ lại lỗi chuyển đổi

Giá giả lập bằng USD/đêm. Quy tắc nguồn: `$` ở đầu, dấu phẩy phân cách hàng nghìn,
dấu chấm phân cách thập phân. Chỉ bỏ khoảng trắng ở hai đầu và các ký hiệu đã biết.
Chuỗi `liên hệ` và `từ $500` có dữ liệu nhưng không theo quy tắc: đó là lỗi chuyển đổi.
`None` là giá trị thiếu từ nguồn.

In [ ]:
gia = pd.Series(["$1,200.00", " $850.00 ", None, "liên hệ", "từ $500"])

# TODO: chuyển sang số; chuỗi không chuyển đổi được nhận giá trị thiếu.
gia_so = ...
# TODO: chỉ đánh dấu chuỗi có dữ liệu nhưng chuyển đổi thất bại.
loi_doc = ...

assert gia_so.iloc[:2].tolist() == [1200.0, 850.0]
assert gia_so.iloc[2:].isna().all()
assert loi_doc.tolist() == [False, False, False, True, True]
assert gia.iloc[0] == "$1,200.00"
print(pd.DataFrame({"gia": gia, "gia_so": gia_so,
                    "loi_doc": loi_doc}))

### Kiểm cả chuỗi, tránh chỉ đọc một phần

Mẫu được cho dưới đây chỉ nhận số không âm: phần nguyên viết liền hoặc có dấu phẩy
sau mỗi ba chữ số; phần thập phân nếu có gồm đúng hai chữ số. Hãy bỏ khoảng trắng
ở hai đầu trước khi kiểm. Không dùng `extract` để kết luận cả chuỗi hợp lệ.

In [ ]:
gia_test = pd.Series(["$2,450.00", " $95 ", "$24,50.00",
                      "khoảng $300.00", "$95.5", None])
PAT_GIA = r"\$(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d{2})?"

# TODO: mask cho biết cả chuỗi có đúng định dạng hay không.
hop_le = ...

assert hop_le.tolist() == [True, True, False, False, False, False]
print(pd.DataFrame({"gia": gia_test, "hop_le": hop_le}))

## Phần 2 · Làm sạch và kiểm quy tắc trên đánh giá thật

Bản đánh giá đầy đủ có cột `comments`; ô dưới chỉ đọc `listing_id` và `comments`.
Nếu có file snapshot ở thư mục `data`, ô dưới đọc bản đã lưu.

In [ ]:
URL_RV = ("https://data.insideairbnb.com/chile/rm/santiago/"
          "2026-06-29/data/reviews.csv.gz")
tep_rv = Path("data/reviews_santiago_full_2026-06-29.csv.gz")
nguon_rv = tep_rv if tep_rv.exists() else URL_RV
rv = pd.read_csv(nguon_rv, usecols=["listing_id", "comments"])
print("Số đánh giá:", len(rv))

### Bước 1 · Khảo sát trước khi xử lý

Tạo `c` từ các đánh giá có nội dung, đo độ dài và xem một số đoạn ngắn.
Giữ chỉ mục để có thể quay lại đúng hàng trong `rv`.

In [ ]:
# TODO: so_thieu là số đánh giá thiếu comments; c gồm các đoạn có nội dung,
# giữ chỉ mục của rv; do_dai là độ dài từng đoạn của c (gợi ý: .str.len()).
so_thieu = ...
c = ...
do_dai = ...

assert len(rv) == 690112
assert so_thieu == 36 and len(c) == 690076
assert c.index.equals(rv.index[rv["comments"].notna()])
assert do_dai.median() == 115.0
print(f"{so_thieu} đoạn thiếu; độ dài trung vị "
      f"{do_dai.median():.0f} ký tự")
print(c[do_dai < 3].head(5).tolist())

In [ ]:
# TODO: đếm các đoạn ngắn hơn 20 ký tự.
so_ngan = ...

assert so_ngan == 58987
print(f"{so_ngan:,}/{len(c):,} đoạn có độ dài < 20")
print(c[do_dai < 20].sample(n=5, random_state=7).tolist())

Độ dài là thông tin khảo sát. Không phải đoạn ngắn nào cũng vô ích:
`Great host!` vẫn có thông tin. Quy tắc loại bỏ dữ liệu cần bám câu hỏi phân tích.

Tuỳ chọn: đọc lại `comments` với `keep_default_na=False` (Bài 6) để xem 36 đánh giá thiếu này
thật sự chứa gì: ô trống, hay một chuỗi mà `read_csv` mặc định đọc thành giá trị thiếu.

### Bước 2 · Thay đúng đoạn HTML đã biết

Nguồn lưu một số chỗ xuống dòng bằng `<br/>`. Thay đúng chuỗi này bằng một dấu cách,
giữ cột gốc để đối chiếu. Đây không phải bộ làm sạch mọi dạng HTML.

In [ ]:
# TODO: so_br là số đoạn chứa <br/>; c_sach là c sau khi thay mỗi <br/>
# bằng một dấu cách.
so_br = ...
c_sach = ...

assert so_br == 116471
assert len(c_sach) == len(c) and c_sach.index.equals(c.index)
assert c_sach.str.contains("<br/>", regex=False, na=False).sum() == 0
assert c.str.contains("<br/>", regex=False, na=False).sum() == so_br
# <br/> có 5 ký tự: mỗi lần thay bằng một dấu cách giảm 4 ký tự và thêm 1 dấu cách.
co_br = c.str.contains("<br/>", regex=False)
so_lan = c[co_br].str.count("<br/>").sum()
assert (c.str.len().sum() - c_sach.str.len().sum() == 4 * so_lan
        and c_sach[co_br].str.count(" ").sum()
        == c[co_br].str.count(" ").sum() + so_lan), "Chỉ thay <br/> bằng một dấu cách."
print(f"Đã thay <br/> trong {so_br:,} đoạn; giữ nguyên bản gốc")

### Bước 3 · Lấy từ đứng sau near hoặc cerca de

Quy tắc dưới lấy **từ đầu tiên** sau `cerca de`, `cerca de la`, `cerca del`,
`near`, `near the` (không phân biệt hoa thường). `\w+` khớp một dãy ký tự từ.
Chạy trên bảng nhỏ trước, rồi mới áp lên cả cột.

In [ ]:
PAT_GAN = r"\b(?:cerca de(?: la|l)?|near(?: the)?)\s+(\w+)"
mau = pd.Series(["near the metro", "cerca del parque",
                 "cerca de todo", "Casa centro"])
print(mau.str.extract(PAT_GAN, flags=re.I, expand=False))

Áp quy tắc lên `c_sach` bằng `str.extract` với `flags=re.I` như ô trên.
Kết quả nên gọi là `tu_sau_gan`, chưa gọi là `dia_diem`.
`so_khop` là số dòng lấy được một từ.

In [ ]:
# TODO: áp quy tắc lên c_sach, giữ chỉ mục; so_khop là số dòng có kết quả.
tu_sau_gan = ...
so_khop = ...

assert tu_sau_gan.index.equals(c_sach.index)
assert so_khop == 36613, "so_khop là số dòng str.extract (flags=re.I) lấy được một từ."
dem_tu = tu_sau_gan.str.lower().value_counts()
print(dem_tu.head(20))

**Đọc nguyên văn.** Trong 20 từ ở ô trên, chọn ít nhất hai từ không phải tên địa điểm
và ghi vào `nghi_van`. Ô dưới in ba đánh giá chứa mỗi từ để bạn kiểm.
Số dòng khớp chỉ cho biết mức bao phủ của quy tắc, chưa đo độ chính xác.

In [ ]:
# TODO: ít nhất hai từ khác nhau (viết thường, lấy từ bảng trên)
# không phải tên địa điểm.
nghi_van = ...

assert len(set(nghi_van)) >= 2 and set(nghi_van) <= set(dem_tu.head(20).index), (
    "Chọn ít nhất hai từ khác nhau, viết thường, trong bảng 20 từ ở ô trên.")
can_xem = tu_sau_gan.str.lower().isin(nghi_van)
print(f"{can_xem.sum():,}/{so_khop:,} dòng khớp cho ra các từ đã chọn")
for tu in nghi_van:
    mau_tu = c_sach[tu_sau_gan.str.lower().eq(tu)].sample(n=3, random_state=7)
    print(f"\n== {tu}")
    for van_ban in mau_tu:
        print("-", van_ban)

Viết nhận xét của bạn ở đây: …

**Đối chiếu sau khi viết nhận xét.** `todo` trong `cerca de todo` nghĩa là “gần mọi thứ”,
không phải tên địa điểm. Quy tắc chỉ lấy **một từ** đứng ngay sau cụm chỉ vị trí:
`cerca de muchos lugares` cho `muchos`, `cerca de los malls` cho `los`,
`cerca del Movistar Arena` chỉ cho `Movistar`. Vì vậy kết quả được gọi là
`tu_sau_gan`, không phải `dia_diem`.

## Phần 3 · Tính tiện nghi theo chỗ ở

Các ô dưới dùng `amenities` của bảng chỗ ở đầy đủ. Mục tiêu là tỷ lệ chỗ ở
**liệt kê đúng mục `Wifi`**. Không suy ra mạng hoạt động tốt từ một mục được liệt kê.

In [ ]:
URL_LS = ("https://data.insideairbnb.com/chile/rm/santiago/"
          "2026-06-29/data/listings.csv.gz")
tep_ls = Path("data/listings_santiago_full_2026-06-29.csv.gz")
nguon_ls = tep_ls if tep_ls.exists() else URL_LS
ls = pd.read_csv(nguon_ls, usecols=["id", "amenities"]).set_index("id")
print("Số chỗ ở:", len(ls))

### Bước 4 · Đọc JSON rồi tách thành từng mục

Dùng `json.loads` trên từng ô, rồi `explode`. Giữ mã chỗ ở làm chỉ mục.
Chuỗi JSON có thể chứa dấu phẩy ngay trong một tên tiện nghi.

In [ ]:
# TODO: ds là Series các danh sách Python; muc là các mục sau khi tách, bỏ ô thiếu.
ds = ...
muc = ...

assert ds.index.equals(ls.index)
assert ds.map(type).eq(list).all()
assert ds.map(len).eq(0).sum() == 14
assert muc.notna().all() and muc.index.isin(ls.index).all(), (
    "muc phải bỏ giá trị thiếu và giữ mã chỗ ở làm chỉ mục.")
assert muc.str.contains(",", regex=False).any(), (
    "Có tên tiện nghi chứa dấu phẩy: đừng tách chuỗi theo dấu phẩy.")
assert muc.eq("Wifi").sum() == 16772
print(muc.value_counts().head(5))

### Bước 5 · Giữ đúng mẫu số

Tạo một giá trị True/False cho mỗi chỗ ở, rồi tính tỷ lệ.
Danh sách rỗng là không có mục được liệt kê và vẫn nằm trong mẫu số.
Nếu một nguồn có `amenities` thiếu, đó là **chưa biết**, cần quy ước riêng.

In [ ]:
# TODO: co_wifi cho biết từng chỗ ở có liệt kê mục "Wifi" hay không;
# ty_le_wifi là tỷ lệ chỗ ở liệt kê Wifi.
co_wifi = ...
ty_le_wifi = ...

assert len(co_wifi) == len(ls)
assert co_wifi.index.is_unique and set(co_wifi.index) == set(ls.index)
assert co_wifi.sum() == 16772
assert abs(ty_le_wifi - 16772 / 18534) < 1e-12
print(f"{co_wifi.sum():,}/{len(ls):,} chỗ ở liệt kê Wifi "
      f"({ty_le_wifi:.1%})")

### Bước 6 · Phân biệt danh sách rỗng với dữ liệu thiếu

Bảng nhỏ giả lập có năm chỗ ở: một danh sách lặp `Wifi`, một danh sách không có `Wifi`,
một danh sách rỗng, một giá trị thiếu và một danh sách chỉ có `Wifi`.
`[]` là biết không có mục được liệt kê; `None` là chưa biết danh sách.
Tính tỷ lệ chỗ ở có Wifi **trong nhóm đã có thông tin tiện nghi**.

In [ ]:
ds_nho = pd.Series([["Wifi", "Wifi", "Kitchen"],
                    ["Kitchen"], [], None, ["Wifi"]],
                   index=["P1", "P2", "P3", "P4", "P5"])
# TODO: so_da_biet là số chỗ ở đã có thông tin tiện nghi;
# ty_le_nho là tỷ lệ chỗ ở có Wifi trong nhóm đó.
so_da_biet = ...
ty_le_nho = ...

assert so_da_biet == 4
assert abs(ty_le_nho - 1 / 2) < 1e-12
print(f"Tỷ lệ đúng: {ty_le_nho:.1%}")

## Phần 4 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

### Tự làm 1 · Kiểm một kết luận từ từ khoá

Một câu trả lời của AI: “Đếm các đánh giá chứa `wifi|internet` là đếm những khách hài lòng
về mạng.” Tạo mask từ khoá, đọc một mẫu **20 đoạn khớp**, rồi nêu các trường hợp làm
kết luận này sai. Viết lại kết luận sao cho phản ánh đúng phép đếm đã làm.

### Tự làm 2 · Trích diện tích có thêm cách viết

Quy tắc cần nhận `45 m2`, `30m²`, `45.5 m2`, `45,5 m²` và không nhận `30m2x`.
Trích diện tích thành số (m²) và giữ nguyên cột tên gốc. Tự thêm các ví dụ không được khớp
trước khi nhờ AI sửa mẫu.

In [ ]:
# Viết lời giải và cách kiểm chứng của bạn ở đây.

**Khai báo AI (nếu dùng):** công cụ, yêu cầu chính, một kết quả bạn đã tự kiểm. …

## Tóm tắt bài lab

- Quy tắc chuẩn hoá cần nêu rõ; giá chưa biết khác giá bằng 0.
- Tìm một phần khớp không thay thế kiểm cả định dạng.
- Khớp từ không xác minh ngữ nghĩa; xem nguyên văn để tìm lỗi.
- Tỷ lệ tiện nghi cần tính theo chỗ ở, kể cả danh sách rỗng và mục lặp.

Bạn cần giải thích được một trường hợp quy tắc khớp đúng và một trường hợp quy tắc
khớp nhưng chưa đủ để kết luận.

Chạy **Restart session and run all**, rồi kiểm tra: mọi ô code đều có kết quả; các ô
nhận xét, giải thích và khai báo AI (nếu dùng) đã điền đủ.

**Đọc thêm:** [McKinney §7.4](https://wesmckinney.com/book/data-cleaning#text_string_manip),
[pandas: dữ liệu chuỗi](https://pandas.pydata.org/docs/user_guide/text.html).

**Bài sau:** xử lý dữ liệu thời gian.